# 01 — UCI Bike Sharing EDA

Hourly demand for the Capital Bikeshare system, Washington DC, 2011-01-01 → 2012-12-31. ~17.4k rows, weather and calendar covariates pre-aligned.

Goal: characterise seasonality (daily, weekly, yearly), weather sensitivity, and any data-quality oddities **before** building forecasts.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL

from chronos2_assessment.data_loader import load_processed

sns.set_theme(context='notebook', style='whitegrid')
FIG_DIR = Path('../outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = load_processed(Path('../data/processed'))
print(df.shape, df['timestamp'].min(), '→', df['timestamp'].max())
df.head()

## Coverage and gaps

In [ ]:
full_idx = pd.date_range(df['timestamp'].min(), df['timestamp'].max(), freq='h')
missing = full_idx.difference(df['timestamp'])
print(f'expected hours: {len(full_idx)}')
print(f'actual rows:    {len(df)}')
print(f'missing hours:  {len(missing)}  ({len(missing) / len(full_idx):.1%})')
print('first 5 missing:', list(missing[:5]))

**Note**: UCI omits hours with no system activity. We'll reindex to a regular hourly grid during modelling so ARIMA/Prophet don't choke on gaps.

## Summary statistics

In [ ]:
df[['target', 'temp', 'atemp', 'hum', 'windspeed']].describe().round(3)

## Full series + last 28 days (the held-out test window)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(df['timestamp'], df['target'], lw=0.4)
axes[0].set_title('Hourly rentals — full series')
axes[0].set_ylabel('rentals/h')

tail = df.iloc[-28 * 24:]
axes[1].plot(tail['timestamp'], tail['target'], lw=0.8, color='#c44')
axes[1].set_title('Hourly rentals — final 28 days (test horizon)')
axes[1].set_ylabel('rentals/h')
fig.tight_layout()
fig.savefig(FIG_DIR / '01_series_overview.png', dpi=120)
plt.show()

## Seasonality: hour × weekday heatmap

In [ ]:
pivot = df.pivot_table(index='hour', columns='weekday', values='target', aggfunc='mean')
pivot.columns = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, cmap='magma', ax=ax, cbar_kws={'label': 'mean rentals/h'})
ax.set_title('Mean hourly demand by weekday — commuter peaks visible Mon–Fri')
fig.tight_layout()
fig.savefig(FIG_DIR / '01_hour_weekday_heatmap.png', dpi=120)
plt.show()

## STL decomposition on the daily aggregate

In [ ]:
daily = df.set_index('timestamp')['target'].resample('D').sum()
stl = STL(daily, period=7, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(10, 7)
fig.suptitle('STL decomposition — daily rentals (period=7)', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / '01_stl_decomposition.png', dpi=120)
plt.show()

## Weather sensitivity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, col, label in zip(axes, ['temp', 'hum', 'windspeed'], ['temperature', 'humidity', 'wind speed']):
    ax.scatter(df[col], df['target'], s=2, alpha=0.15)
    ax.set_xlabel(f'{label} (normalised)')
    ax.set_title(f'rentals vs {label}')
axes[0].set_ylabel('rentals/h')
fig.tight_layout()
fig.savefig(FIG_DIR / '01_weather_scatter.png', dpi=120)
plt.show()

In [ ]:
corr = df[['target', 'temp', 'atemp', 'hum', 'windspeed', 'weather']].corr()['target'].drop('target')
print('Pearson correlation with target:')
print(corr.round(3).sort_values(ascending=False))

## Takeaways

- Strong **daily** seasonality with twin commuter peaks on weekdays (~8am, ~5pm); weekends show a single midday peak.
- Clear **yearly** trend: 2012 demand substantially higher than 2011 (system maturing).
- **Temperature** is the dominant weather driver (positive correlation). Humidity is negatively correlated; wind speed weakly so.
- ~1% of hours are missing — reindex during modelling.

These are the patterns Chronos-2 will need to capture zero-shot, and that classical baselines (Prophet, ARIMA) and ML (LightGBM with lag features) can explicitly model.